# IOAI — 2025 Lab Lab1 Finetune Resnet (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-lab-lab1-finetune-resnet/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Lab 1 — Finetune a ResNet (치와와 vs 머핀)

MAIO 2025 TSP **Lab 1**. ImageNet 사전학습 **ResNet** 을 미세조정해 **치와와(chihuahua) vs 머핀(muffin)**
이진분류기를 만든다. 학습 없이는 ImageNet 백본이 두 클래스를 잘 못 가르지만, 마지막 층만 미세조정해도
거의 완벽해진다.

- `data/train/{chihuahua,muffin}/*.jpg` — 라벨된 학습 이미지(각 400장, 160×160)
- `data/test/*.jpg` — held-out 200장(라벨 비공개)

**제출**: 각 test 이미지에 대해 `id,target`(0=chihuahua, 1=muffin) 예측을 `submission.csv` 로 저장.
정확도(accuracy)로 채점한다.

## 셋업

In [ ]:
import os, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 데이터셋 / 데이터로더

In [ ]:
CLASSES = ["chihuahua", "muffin"]
tf = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class DogMuffin(Dataset):
    def __init__(self, files, labels=None):
        self.files, self.labels = files, labels
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        x = tf(Image.open(self.files[i]).convert("RGB"))
        y = -1 if self.labels is None else self.labels[i]
        return x, y

train_files, train_labels = [], []
for c, name in enumerate(CLASSES):
    for f in sorted(glob.glob(f"data/train/{name}/*.jpg")):
        train_files.append(f); train_labels.append(c)
test_files = sorted(glob.glob("data/test/*.jpg"))
test_ids = [os.path.splitext(os.path.basename(f))[0] for f in test_files]
print("train:", len(train_files), "test:", len(test_files))

train_dl = DataLoader(DogMuffin(train_files, train_labels), batch_size=32, shuffle=True)
test_dl  = DataLoader(DogMuffin(test_files), batch_size=32)

## 모델 — 동결 백본 + 선형 헤드
마지막 FC 층만 학습한다(백본 동결).

In [ ]:
m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in m.parameters(): p.requires_grad = False
m.fc = nn.Linear(m.fc.in_features, 2)
m = m.to(device)
opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)
crit = nn.CrossEntropyLoss()

## 학습(약 2 epoch)

In [ ]:
m.train()
for e in range(2):
    tot = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = crit(m(x), y)
        loss.backward(); opt.step()
        tot += loss.item() * len(x)
    print(f"epoch {e} loss {tot/len(train_files):.4f}")

## 예측 → submission.csv

In [ ]:
m.eval()
preds = []
with torch.no_grad():
    for x, _ in test_dl:
        preds += m(x.to(device)).argmax(1).cpu().tolist()
pd.DataFrame({"id": test_ids, "target": preds}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(preds))

베이스라인은 동결 백본 + 선형 헤드만으로도 이미 매우 높은 정확도가 나온다. 더 끌어올리려면 백본을 미세조정하라(모범답안 참고).

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)